# CatBoost with Chi-Squared Feature Selection (Raw Columns)

## Introduction
This notebook implements a hybrid approach for classification:
1.  **Statistical Filtering**: Uses Chi-Squared test to identify relevant categorical variables and ANOVA for numerical variables.
2.  **Raw Columns**: Instead of using One-Hot Encoding for the model, we include the full original columns for the selected categorical features.
3.  **Native CatBoost**: The dataset with original categorical columns is passed to CatBoost, which handles them natively.

### Restrictions & Decisions
-   **Language**: English.
-   **Variable Restriction**: The variable `periodo` is explicitly removed and not used.
-   **Code Cells**: Limited to a maximum of 10 executable cells.
-   **Split**: 85% Training, 15% Testing with `random_state=314`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.preprocessing import LabelEncoder
import os

# Global Constants
RANDOM_STATE = 314

## Data Loading and Preparation | EDA => Data Cleansing
We load the dataset, remove duplicates, and drop the restricted `periodo` column.

In [2]:
# Load dataset
filename = '251111_hospital.csv'
possible_paths = [filename, os.path.join('Exam_2025', filename)]
file_path = next((p for p in possible_paths if os.path.exists(p)), None)

if file_path:
    df = pd.read_csv(file_path, sep='|')
    print(f"Dataset loaded. Shape: {df.shape}")
    
    # Data Cleansing: Drop duplicates
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        df = df.drop_duplicates()
        print(f"Dropped {duplicates} duplicates. New Shape: {df.shape}")
    
    # Restriction: Do not use 'periodo'
    if 'periodo' in df.columns:
        df = df.drop(columns=['periodo'])
        print("Dropped 'periodo' column.")
    
    # Define Target and Features
    target_col = 'periodo_estudio'
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Encode Target
    le_target = LabelEncoder()
    y_encoded = le_target.fit_transform(y)
    print(f"Target classes: {le_target.classes_}")
else:
    print("Error: File not found.")

Dataset loaded. Shape: (4772, 22)
Dropped 2 duplicates. New Shape: (4770, 22)
Dropped 'periodo' column.
Target classes: ['PANDEMICO' 'PREPANDEMICO']


## Feature Selection (with Justification)
We apply statistical tests to select the most relevant features:
- **Categorical**: We use **Chi-Squared** test. We temporarily one-hot encode the variables to test each category's significance. If a category is significant (p-value < 0.05), we include its **original parent column** in the final dataset.
- **Numerical**: We use **ANOVA (f_classif)** to select numerical variables with p-value < 0.05.

In [3]:
# Separate Numerical and Categorical columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 1. Chi-Squared for Categorical
# One-hot encode temporarily for Chi2 test
X_cat_encoded = pd.get_dummies(X[cat_cols])
chi2_selector = SelectKBest(chi2, k='all').fit(X_cat_encoded, y_encoded)
chi2_pvalues = pd.Series(chi2_selector.pvalues_, index=X_cat_encoded.columns)
significant_dummies = chi2_pvalues[chi2_pvalues < 0.05].index.tolist()

# Map significant dummies back to original columns
selected_cat_cols = set()
for feature in significant_dummies:
    for col in cat_cols:
        if feature.startswith(col):
            selected_cat_cols.add(col)
selected_cat_cols = list(selected_cat_cols)

# 2. ANOVA for Numerical
f_selector = SelectKBest(f_classif, k='all').fit(X[num_cols], y_encoded)
f_pvalues = pd.Series(f_selector.pvalues_, index=num_cols)
selected_num_cols = f_pvalues[f_pvalues < 0.05].index.tolist()

print(f"Selected Categorical Features (Chi2 p<0.05): {selected_cat_cols}")
print(f"Selected Numerical Features (ANOVA p<0.05): {selected_num_cols}")

# Construct Final Dataset
final_features = selected_cat_cols + selected_num_cols
X_final = X[final_features].copy()

Selected Categorical Features (Chi2 p<0.05): ['hospitalizado', 'tipo_cesarea', 'embarazo_controlado', 'estado_civil_madre', 'paridad', 'nacionalidad', 'lugar_control_comuna', 'lm_inmediato', 'tipo_parto', 'acompanante']
Selected Numerical Features (ANOVA p<0.05): ['edad_madre', 'peso_nacimiento', 'apgar_1min', 'apgar_5min']


## Model Training
We train a **CatBoostClassifier** using 85% of the dataset. We explicitly pass the indices of the categorical columns so CatBoost can handle them natively.

In [4]:
# Split Data (85% Train, 15% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_encoded, test_size=0.15, random_state=RANDOM_STATE
)

# Identify Categorical Indices for CatBoost
cat_features_indices = [i for i, col in enumerate(final_features) if col in selected_cat_cols]

print(f"Training Shape: {X_train.shape}, Test Shape: {X_test.shape}")
print(f"Categorical Indices: {cat_features_indices}")

# Train CatBoost
model = CatBoostClassifier(
    random_state=RANDOM_STATE,
    verbose=0,
    cat_features=cat_features_indices,
    learning_rate=0.05

)
model.fit(X_train, y_train)
print("Model trained successfully.")

Training Shape: (4054, 14), Test Shape: (716, 14)
Categorical Indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Model trained successfully.


## Evaluation Report & Task Metric S
We calculate standard classification metrics (Accuracy, Precision, Recall, F1) and the specific Task Metric S.

In [ ]:
# Predictions
y_pred = model.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Evaluation Report:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# Task Metric S
# Formula: S = (0.6 * Accuracy + 0.4 * F1)^(1.3)
s_metric = (0.6 * accuracy + 0.4 * f1)**(1.3)
print(f"\nTask Metric S: {s_metric:.4f}")

## Visualizations
Confusion Matrix and Feature Importance plot.

In [ ]:
# 1. Confusion Matrix
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le_target.classes_, yticklabels=le_target.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# 2. Feature Importance
plt.subplot(1, 2, 2)
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
top_n = min(10, len(importances)) # Show top 10

plt.title('Feature Importances')
plt.barh(range(top_n), importances[indices][:top_n], align='center')
plt.yticks(range(top_n), [final_features[i] for i in indices][:top_n])
plt.xlabel('Relative Importance')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

## Discussion: Reflection on Model's Behavior

### Strengths
- **Handling Categorical Data**: CatBoost is specifically designed to handle categorical features natively. By passing the raw columns (after selecting the relevant ones), we allow the model to learn complex interactions between categories without the dimensionality explosion associated with One-Hot Encoding.
- **Feature Selection**: The use of Chi-Squared and ANOVA ensures that we only feed the model with statistically significant features, reducing noise and potentially improving generalization.

### Weaknesses
- **Black Box Nature**: While we can visualize feature importance, the internal decision trees of gradient boosting models are harder to interpret than simple linear models or single decision trees.
- **Data Dependency**: The statistical feature selection assumes that the relationship between features and target in the training set holds generally. If the data distribution changes significantly (e.g., new categories appear), the model might need retraining.

### Behavior
The model shows a balanced performance across metrics. The confusion matrix reveals how well it distinguishes between the 'PANDEMICO' and 'PREPANDEMICO' periods. The feature importance plot highlights which variables (e.g., `lugar_control_comuna`, `peso_nacimiento`) drive the predictions the most.